In [ ]:
"""
PIPELINE STAGE 5: Hallucination Candidate Extraction
----------------------------------------------------
Role in Thesis:
    Operationalizes clinically relevant mismatch patterns. Extracts observations 
    that appear *only* in the generated output (H_i = G_i \ R_i) as candidates 
    for unsupported findings (hallucinations).
    
Review Integration:
    Prepares a clean template for manual expert evaluation, labeling candidates 
    as 'supported', 'unsupported', or 'uncertain'. These labels act as the bridge 
    between automated extraction and probabilistic downstream modeling.
"""

In [ ]:
# =====================================================================
# PREPARING OBSERVATION-LEVEL REVIEW TEMPLATE
# =====================================================================
# Instead of scoring full reports, we extract the precise generated-only 
# observations (H_i) for targeted manual expert review. This reduces cognitive 
# load and links the automated extraction back to a human interpretive layer.

# =====================================================================
# REINTEGRATING MANUAL LABELS & DEFINING HALLUCINATION RATES
# =====================================================================
# Mocks/Integrates the evaluated manual labels ('supported', 'unsupported', 'uncertain').
# 
# Operational Definitions for Sensitivity Analysis:
# - STRICT Formulation: Counts only explicitly 'unsupported' claims as hallucinations.
# - LENIENT Formulation: Additionally counts 'uncertain' additions as hallucinations.
# These distinct operationalizations are passed to the downstream regression.

import os
import sys

os.environ.setdefault("PYTENSOR_FLAGS", "cxx=,optimizer_excluding=fusion")

!{sys.executable} -m pip install -q pandas numpy

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
CASE_DATAPATH = PROJECT_ROOT / "outputs" / "radgraph_analysis_1500_subset" / "case_level_comparison.csv"
REVIEW_DATAPATH = PROJECT_ROOT / "outputs" / "manual_review_1500_subset" / "manual_review_template_filled.csv"
OUT = PROJECT_ROOT / "outputs" / "manual_review_1500_subset" / "final_hallucination_outputs"
OUT.mkdir(parents=True, exist_ok=True)

print("Current working directory:", Path.cwd())
print("CASE_DATAPATH:", CASE_DATAPATH)
print("Resolved CASE_DATAPATH:", CASE_DATAPATH.resolve())
print("CASE_DATAPATH exists:", CASE_DATAPATH.exists())

print("REVIEW_DATAPATH:", REVIEW_DATAPATH)
print("Resolved REVIEW_DATAPATH:", REVIEW_DATAPATH.resolve())
print("REVIEW_DATAPATH exists:", REVIEW_DATAPATH.exists())

if not CASE_DATAPATH.exists():
    raise FileNotFoundError(f"Could not find {CASE_DATAPATH}")

if not REVIEW_DATAPATH.exists():
    raise FileNotFoundError(
        f"Could not find {REVIEW_DATAPATH}. "
        "Save your labeled review file under this name or adjust REVIEW_DATAPATH."
    )

# ------------------------------------------------------------------
# Load data
# ------------------------------------------------------------------
case_df = pd.read_csv(CASE_DATAPATH)
review_df = pd.read_csv(REVIEW_DATAPATH)

print("Loaded case_df shape:", case_df.shape)
print("Loaded review_df shape:", review_df.shape)

display(case_df.head())
display(review_df.head())

# ------------------------------------------------------------------
# Normalize columns
# ------------------------------------------------------------------
case_rename_map = {
    "caseid": "case_id",
    "generatedreport": "generated_report",
    "referencereport": "reference_report",
    "generatedobservationcount": "generated_observation_count",
    "referenceobservationcount": "reference_observation_count",
    "sharedobservationcount": "shared_observation_count",
    "generatedonlycount": "generated_only_count",
    "referenceonlycount": "reference_only_count",
    "precisionobs": "precision_obs",
    "recallobs": "recall_obs",
    "f1obs": "f1_obs",
    "jaccardobs": "jaccard_obs",
    "generatedonly": "generated_only",
    "referenceonly": "reference_only",
}

review_rename_map = {
    "caseid": "case_id",
    "reviewitemid": "review_item_id",
    "manuallabel": "manual_label",
    "reviewdate": "review_date",
}

case_df = case_df.rename(columns={c: case_rename_map.get(c, c) for c in case_df.columns})
review_df = review_df.rename(columns={c: review_rename_map.get(c, c) for c in review_df.columns})

case_df["case_id"] = pd.to_numeric(case_df["case_id"], errors="coerce").astype("Int64")
review_df["case_id"] = pd.to_numeric(review_df["case_id"], errors="coerce").astype("Int64")

# ------------------------------------------------------------------
# Validate manual labels
# ------------------------------------------------------------------
review_df["manual_label"] = (
    review_df["manual_label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

allowed_labels = {"unsupported", "supported", "uncertain"}
found_labels = set(v for v in review_df["manual_label"].unique() if v != "")
invalid_labels = sorted(found_labels - allowed_labels)

print("Found labels:", sorted(found_labels))

if invalid_labels:
    raise ValueError(f"Invalid manual labels found: {invalid_labels}")

unlabeled_rows = review_df["manual_label"].eq("").sum()
print("Unlabeled rows:", int(unlabeled_rows))

if unlabeled_rows > 0:
    raise ValueError(
        f"There are {int(unlabeled_rows)} unlabeled rows. "
        "Please complete manual_label before creating the final hallucination sets."
    )

# ------------------------------------------------------------------
# Derive strict and lenient hallucination flags
# ------------------------------------------------------------------
review_df["final_hallucination_strict"] = (review_df["manual_label"] == "unsupported").astype(int)
review_df["final_hallucination_lenient"] = (
    review_df["manual_label"].isin(["unsupported", "uncertain"])
).astype(int)

# ------------------------------------------------------------------
# Save labeled review master
# ------------------------------------------------------------------
review_df.to_csv(OUT / "manual_review_labeled_master.csv", index=False)

# ------------------------------------------------------------------
# Row-level final sets
# ------------------------------------------------------------------
strict_set_df = review_df.loc[review_df["final_hallucination_strict"] == 1].copy()
lenient_set_df = review_df.loc[review_df["final_hallucination_lenient"] == 1].copy()

strict_set_df.to_csv(OUT / "final_hallucination_set_strict.csv", index=False)
lenient_set_df.to_csv(OUT / "final_hallucination_set_lenient.csv", index=False)

# ------------------------------------------------------------------
# Case-level aggregation
# ------------------------------------------------------------------
case_level_df = (
    review_df.groupby("case_id")
    .agg(
        n_reviewed=("review_item_id", "count"),
        n_supported=("manual_label", lambda s: int((s == "supported").sum())),
        n_unsupported=("manual_label", lambda s: int((s == "unsupported").sum())),
        n_uncertain=("manual_label", lambda s: int((s == "uncertain").sum())),
        hallucination_binary_strict=("final_hallucination_strict", "max"),
        hallucination_binary_lenient=("final_hallucination_lenient", "max"),
        hallucination_count_strict=("final_hallucination_strict", "sum"),
        hallucination_count_lenient=("final_hallucination_lenient", "sum"),
    )
    .reset_index()
)

case_level_df["hallucination_rate_strict"] = (
    case_level_df["hallucination_count_strict"] / case_level_df["n_reviewed"]
)
case_level_df["hallucination_rate_lenient"] = (
    case_level_df["hallucination_count_lenient"] / case_level_df["n_reviewed"]
)

# ------------------------------------------------------------------
# Merge with case-level comparison file
# ------------------------------------------------------------------
merge_cols = [
    "case_id",
    "generated_observation_count",
    "reference_observation_count",
    "shared_observation_count",
    "generated_only_count",
    "reference_only_count",
    "precision_obs",
    "recall_obs",
    "f1_obs",
    "jaccard_obs",
]

merge_cols = [c for c in merge_cols if c in case_df.columns]

case_level_regression_df = case_level_df.merge(
    case_df[merge_cols],
    on="case_id",
    how="left",
)

case_level_regression_df.to_csv(
    OUT / "final_hallucination_case_level.csv",
    index=False,
)

# ------------------------------------------------------------------
# Summary tables
# ------------------------------------------------------------------
label_distribution_df = (
    review_df["manual_label"]
    .value_counts(dropna=False)
    .rename_axis("manual_label")
    .reset_index(name="n")
)
label_distribution_df.to_csv(OUT / "manual_label_distribution.csv", index=False)

summary_df = pd.DataFrame(
    [
        {
            "n_review_rows": len(review_df),
            "n_review_cases": review_df["case_id"].nunique(),
            "n_strict_hallucinations": len(strict_set_df),
            "n_lenient_hallucinations": len(lenient_set_df),
            "n_case_level_rows": len(case_level_regression_df),
            "strict_case_binary_positive": int(case_level_regression_df["hallucination_binary_strict"].sum()),
            "lenient_case_binary_positive": int(case_level_regression_df["hallucination_binary_lenient"].sum()),
            "mean_strict_rate": float(case_level_regression_df["hallucination_rate_strict"].mean()),
            "mean_lenient_rate": float(case_level_regression_df["hallucination_rate_lenient"].mean()),
        }
    ]
)
summary_df.to_csv(OUT / "final_hallucination_summary.csv", index=False)

# ------------------------------------------------------------------
# Output preview
# ------------------------------------------------------------------
print("\nSaved outputs to:", OUT.resolve())
print("\nMain files:")
print("-", OUT / "manual_review_labeled_master.csv")
print("-", OUT / "final_hallucination_set_strict.csv")
print("-", OUT / "final_hallucination_set_lenient.csv")
print("-", OUT / "final_hallucination_case_level.csv")
print("-", OUT / "manual_label_distribution.csv")
print("-", OUT / "final_hallucination_summary.csv")

display(label_distribution_df)
display(summary_df)
display(case_level_regression_df.head(10))
display(strict_set_df.head(10))
display(lenient_set_df.head(10))


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Current working directory: /workspace/thesis/notebooks
CASE_DATAPATH: /workspace/thesis/outputs/radgraph_analysis_1500_subset/case_level_comparison.csv
Resolved CASE_DATAPATH: /workspace/thesis/outputs/radgraph_analysis_1500_subset/case_level_comparison.csv
CASE_DATAPATH exists: True
REVIEW_DATAPATH: /workspace/thesis/outputs/manual_review_1500_subset/manual_review_template_filled.csv
Resolved REVIEW_DATAPATH: /workspace/thesis/outputs/manual_review_1500_subset/manual_review_template_filled.csv
REVIEW_DATAPATH exists: True
Loaded case_df shape: (272, 15)
Loaded review_df shape: (678, 21)


,case_id,generated_report,reference_report,generated_observation_count,reference_observation_count,shared_observation_count,generated_only_count,reference_only_count,precision_obs,recall_obs,f1_obs,jaccard_obs,shared_observations,generated_only,reference_only
0,1041064153850317,AP and lateral views of the chest. There is a ...,Large right-sided pleural effusion with underl...,5,4,2,3,2,0.400000,0.50,0.444444,0.285714,large effusion; small effusion,mild congestion; pneumothorax; unchanged,atelectasis; consolidation
1,1102224550126222,Patient is status post median sternotomy and a...,Slight improvement in mild pulmonary edema. Pa...,13,4,3,10,1,0.230769,0.75,0.352941,0.214286,atelectasis; infection; patchy opacities,acute abnormalities; aspiration; large effusio...,slight improvement mild edema
2,1156909359234239,The ET tube is 5 cm above the carina. The NG t...,1. ET tube terminating 5.1 cm above the carina...,8,4,2,6,2,0.250000,0.50,0.333333,0.200000,elevation; et tube,effusion; haze; increased infiltrate; ng tube ...,orogastric tube; worsening mild - to - moderat...
3,1160762850790949,The endotracheal tube is 4.5 cm above the cari...,1. Endotracheal tube appropriately retracted t...,8,4,2,6,2,0.250000,0.50,0.333333,0.200000,edema; endotracheal tube,catheter; effusion; enteric tube tip; moderate...,mild cardiomegaly; moderate greater effusions
4,1188092357045176,The lungs are clear. The cardiomediastinal sil...,Tiny right pleural effusion.,4,2,1,3,1,0.250000,0.50,0.333333,0.200000,effusion,clear; normal; pneumothorax,tiny


,review_item_id,case_id,auto_subset,observation,located_at,suggestive_of,tags,generated_only_count,reference_only_count,precision_obs,...,generated_report,reference_report,generated_only,reference_only,manual_label,reviewer,review_date,notes,final_hallucination_strict,final_hallucination_lenient
0,MR_0001,1004616657379357,generated_only_candidate,wire fractured,NaN,NaN,definitely present,8,2,0.0,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,uncertain,Joel Kakkattu,"March 24th, 2026",The finding is extra or more specific than the...,0,1
1,MR_0002,1004616657379357,generated_only_candidate,normal,heart size,NaN,definitely present,8,2,0.0,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,uncertain,Joel Kakkattu,"March 24th, 2026",The finding is extra or more specific than the...,0,1
2,MR_0003,1004616657379357,generated_only_candidate,unremarkable,mediastinal and hilar contours,NaN,definitely present,8,2,0.0,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,unsupported,Joel Kakkattu,"March 24th, 2026",The generated normal statement conflicts with ...,1,1
3,MR_0004,1004616657379357,generated_only_candidate,normal,pulmonary vasculature,NaN,definitely present,8,2,0.0,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,unsupported,Joel Kakkattu,"March 24th, 2026",The generated normal statement conflicts with ...,1,1
4,MR_0005,1004616657379357,generated_only_candidate,calcified granuloma,right lower lobe,NaN,definitely present,8,2,0.0,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,uncertain,Joel Kakkattu,"March 24th, 2026",The finding is extra or more specific than the...,0,1


Found labels: ['supported', 'uncertain', 'unsupported']
Unlabeled rows: 0

Saved outputs to: /workspace/thesis/outputs/manual_review_1500_subset/final_hallucination_outputs

Main files:
- /workspace/thesis/outputs/manual_review_1500_subset/final_hallucination_outputs/manual_review_labeled_master.csv
- /workspace/thesis/outputs/manual_review_1500_subset/final_hallucination_outputs/final_hallucination_set_strict.csv
- /workspace/thesis/outputs/manual_review_1500_subset/final_hallucination_outputs/final_hallucination_set_lenient.csv
- /workspace/thesis/outputs/manual_review_1500_subset/final_hallucination_outputs/final_hallucination_case_level.csv
- /workspace/thesis/outputs/manual_review_1500_subset/final_hallucination_outputs/manual_label_distribution.csv
- /workspace/thesis/outputs/manual_review_1500_subset/final_hallucination_outputs/final_hallucination_summary.csv


,manual_label,n
0,uncertain,555
1,supported,73
2,unsupported,50


,n_review_rows,n_review_cases,n_strict_hallucinations,n_lenient_hallucinations,n_case_level_rows,strict_case_binary_positive,lenient_case_binary_positive,mean_strict_rate,mean_lenient_rate
0,678,95,50,605,95,34,95,0.086157,0.887857


,case_id,n_reviewed,n_supported,n_unsupported,n_uncertain,hallucination_binary_strict,hallucination_binary_lenient,hallucination_count_strict,hallucination_count_lenient,hallucination_rate_strict,hallucination_rate_lenient,generated_observation_count,reference_observation_count,shared_observation_count,generated_only_count,reference_only_count,precision_obs,recall_obs,f1_obs,jaccard_obs
0,1004616657379357,9,0,4,5,1,1,4,9,0.444444,1.000000,8,2,0,8,2,0.000000,0.000000,NaN,0.000000
1,1026887750239281,9,4,1,4,1,1,1,5,0.111111,0.555556,11,7,2,9,5,0.181818,0.285714,0.222222,0.125000
2,1026887751513702,6,0,0,6,0,1,0,6,0.000000,1.000000,7,2,1,6,1,0.142857,0.500000,0.222222,0.125000
3,1026887754137212,5,0,0,5,0,1,0,5,0.000000,1.000000,6,4,1,5,3,0.166667,0.250000,0.200000,0.111111
4,1026887757976739,6,0,0,6,0,1,0,6,0.000000,1.000000,6,3,0,6,3,0.000000,0.000000,NaN,0.000000
5,1040237251966612,7,0,0,7,0,1,0,7,0.000000,1.000000,7,1,0,7,1,0.000000,0.000000,NaN,0.000000
6,1040237256446284,5,0,0,5,0,1,0,5,0.000000,1.000000,5,2,0,5,2,0.000000,0.000000,NaN,0.000000
7,1040237259239338,7,0,0,7,0,1,0,7,0.000000,1.000000,7,2,0,7,2,0.000000,0.000000,NaN,0.000000
8,1041064153850317,3,0,0,3,0,1,0,3,0.000000,1.000000,5,4,2,3,2,0.400000,0.500000,0.444444,0.285714
9,1041064156839020,4,2,1,1,1,1,1,2,0.250000,0.500000,4,3,0,4,3,0.000000,0.000000,NaN,0.000000


,review_item_id,case_id,auto_subset,observation,located_at,suggestive_of,tags,generated_only_count,reference_only_count,precision_obs,...,generated_report,reference_report,generated_only,reference_only,manual_label,reviewer,review_date,notes,final_hallucination_strict,final_hallucination_lenient
2,MR_0003,1004616657379357,generated_only_candidate,unremarkable,mediastinal and hilar contours,NaN,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,unsupported,Joel Kakkattu,"March 24th, 2026",The generated normal statement conflicts with ...,1,1
3,MR_0004,1004616657379357,generated_only_candidate,normal,pulmonary vasculature,NaN,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,unsupported,Joel Kakkattu,"March 24th, 2026",The generated normal statement conflicts with ...,1,1
5,MR_0006,1004616657379357,generated_only_candidate,clear,NaN,NaN,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,unsupported,Joel Kakkattu,"March 24th, 2026",The generated normal statement conflicts with ...,1,1
8,MR_0009,1004616657379357,generated_only_candidate,acute abnormalities,osseous,NaN,definitely absent,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,unsupported,Joel Kakkattu,"March 24th, 2026",The generated negative/normal statement confli...,1,1
16,MR_0017,1026887750239281,generated_only_candidate,large effusion,pleural,NaN,definitely absent,9,5,0.181818,...,Single AP upright portable view of the chest w...,1. Left PICC tip appears to terminate in the d...,consolidation; enlargement; large effusion; li...,aeration; effusion; left picc; residual streak...,unsupported,Joel Kakkattu,"March 24th, 2026",The generated absence statement conflicts with...,1,1
58,MR_0059,1041064156839020,generated_only_candidate,clear,upper lung zones,NaN,definitely present,4,3,0.000000,...,"There are large bilateral pleural effusions, l...",Moderate bilateral pleural effusions with poss...,clear; effusions; large effusions greater; pne...,disease; moderate effusions; underlying,unsupported,Joel Kakkattu,"March 24th, 2026",The generated normal statement conflicts with ...,1,1
97,MR_0098,1043978156498272,generated_only_candidate,acute abnormalities,osseous,NaN,definitely absent,9,9,0.000000,...,Left-sided Port-A-Cath tip terminates in the S...,Mild pulmonary edema superimposed on known lun...,acute abnormalities; diffusely calcified; effu...,cardiomegaly; chronic; displaced; fibrosis; fr...,unsupported,Joel Kakkattu,"March 24th, 2026",The generated negative/normal statement confli...,1,1
119,MR_0120,1044929754773340,generated_only_candidate,acute abnormality,osseous,NaN,definitely absent,6,1,0.000000,...,AP and lateral views of the chest. Lower lung ...,Pulmonary edema.,acute abnormality; confluent consolidation; ef...,edema,unsupported,Joel Kakkattu,"March 24th, 2026",The generated negative/normal statement confli...,1,1
139,MR_0140,1071547752467293,generated_only_candidate,acute abnormalities,osseous,NaN,definitely absent,10,3,0.000000,...,The patient is status post median sternotomy a...,Stable cardiomegaly without signs of pneumonia...,acute abnormalities; congestion; diffusely cal...,chf; pneumonia; stable cardiomegaly,unsupported,Joel Kakkattu,"March 24th, 2026",The generated negative/normal statement confli...,1,1
144,MR_0145,1086720251707133,generated_only_candidate,acute abnormality,osseous,NaN,definitely absent,5,3,0.000000,...,AP and lateral views of the chest. There are d...,Find

,review_item_id,case_id,auto_subset,observation,located_at,suggestive_of,tags,generated_only_count,reference_only_count,precision_obs,...,generated_report,reference_report,generated_only,reference_only,manual_label,reviewer,review_date,notes,final_hallucination_strict,final_hallucination_lenient
0,MR_0001,1004616657379357,generated_only_candidate,wire fractured,NaN,NaN,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,uncertain,Joel Kakkattu,"March 24th, 2026",The finding is extra or more specific than the...,0,1
1,MR_0002,1004616657379357,generated_only_candidate,normal,heart size,NaN,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,uncertain,Joel Kakkattu,"March 24th, 2026",The finding is extra or more specific than the...,0,1
2,MR_0003,1004616657379357,generated_only_candidate,unremarkable,mediastinal and hilar contours,NaN,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,unsupported,Joel Kakkattu,"March 24th, 2026",The generated normal statement conflicts with ...,1,1
3,MR_0004,1004616657379357,generated_only_candidate,normal,pulmonary vasculature,NaN,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,unsupported,Joel Kakkattu,"March 24th, 2026",The generated normal statement conflicts with ...,1,1
4,MR_0005,1004616657379357,generated_only_candidate,calcified granuloma,right lower lobe,NaN,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,uncertain,Joel Kakkattu,"March 24th, 2026",The finding is extra or more specific than the...,0,1
5,MR_0006,1004616657379357,generated_only_candidate,clear,NaN,NaN,definitely present,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,unsupported,Joel Kakkattu,"March 24th, 2026",The generated normal statement conflicts with ...,1,1
6,MR_0007,1004616657379357,generated_only_candidate,effusion,pleural,NaN,definitely absent,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,uncertain,Joel Kakkattu,"March 24th, 2026",The absence claim is not explicit in the refer...,0,1
7,MR_0008,1004616657379357,generated_only_candidate,pneumothorax,NaN,NaN,definitely absent,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,uncertain,Joel Kakkattu,"March 24th, 2026",The absence claim is not explicit in the refer...,0,1
8,MR_0009,1004616657379357,generated_only_candidate,acute abnormalities,osseous,NaN,definitely absent,8,2,0.000000,...,The patient is status post median sternotomy a...,No radiographic findings to suggest pneumonia.,acute abnormalities; calcified granuloma; clea...,findings; pneumonia,unsupported,Joel Kakkattu,"March 24th, 2026",The generated negative/normal statement confli...,1,1
11,MR_0012,1026887750239281,generated_only_candidate,enlargement,cardiac silhouette,NaN,definitely present,9,5,0.181818,...,Single AP upright portable view of the chest w...,1. Left PICC tip appears to terminate in the d...,consolidation; enlargement; large effusion; li...,a